In [2]:
import pandas as pd

files = {
    "orders":    "olist_orders_dataset.csv",
    "items":     "olist_order_items_dataset.csv",
    "products":  "olist_products_dataset.csv",
    "sellers":   "olist_sellers_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "cat":       "product_category_name_translation.csv",
}
d = {k: pd.read_csv(v) for k, v in files.items()}


In [3]:
data_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders = d["orders"].copy()
for c in data_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

In [4]:
orders.dtypes 

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [5]:
delivered = orders[
    (orders["order_status"] == "delivered")
    & (orders["order_delivered_customer_date"].notna())
].copy()

print(len(orders), "→", len(delivered), "kept")


99441 → 96470 kept


In [6]:
order_items = d["items"].groupby("order_id").agg(
    item_count=("order_item_id", "count"),
    order_value=("price", "sum"),
    freight_cost=("freight_value", "sum"),
    seller_id=("seller_id", "first"),
    product_id=("product_id", "first"),
).reset_index()

print(len(order_items), "unique orders after grouping items")


98666 unique orders after grouping items


In [7]:
df = (delivered
      .merge(order_items, on="order_id", how="inner")
      .merge(d["customers"], on="customer_id", how="left")
      .merge(d["sellers"], on="seller_id", how="left")
      .merge(d["products"][["product_id","product_category_name","product_weight_g"]],
             on="product_id", how="left")
      .merge(d["cat"], on="product_category_name", how="left"))

print(len(df), "rows after join")


96470 rows after join


In [8]:
assert df["order_id"].is_unique, "orders duplicated - the join broke"
print("all good, one row per order")


all good, one row per order


In [9]:
df["category"] = df["product_category_name_english"].fillna(
    df["product_category_name"]).fillna("unknown")

print(df["category"].isna().sum(), "missing categories after fallback")


0 missing categories after fallback


In [10]:
df["delivery_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

df["promised_days"] = (
    df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
).dt.days

df["delay_days"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

df["on_time"]     = (df["delay_days"] <= 0).astype(int)
df["freight_pct"] = 100 * df["freight_cost"] / df["order_value"].replace(0, pd.NA)
df["cross_state"] = (df["seller_state"] != df["customer_state"]).astype(int)

print("on-time rate:", round(100 * df["on_time"].mean(), 1), "%")
print("avg delivery days:", round(df["delivery_days"].mean(), 1))


on-time rate: 93.2 %
avg delivery days: 12.1


In [11]:
df.to_csv("delivery_clean.csv", index=False)
print("saved delivery_clean.csv:", len(df), "rows")


saved delivery_clean.csv: 96470 rows
